# Model 6 · species-aware joint ranking
Emad. Starting point: Dalvin’s Model 3d, reported public Kaggle MAP@5 0.85960. This experiment retains its PCA/LDA, identity similarity summaries, viewpoint and mutual RootSIFT evidence. It tests the PDF’s direct ranking, shift measurement and identity-disjoint evaluation ideas. A ridge flank map contributes one additional retrieval slot. Compute is concentrated on the nearest raw and LDA gallery sightings for each query. This budget changes verification coverage relative to 3d. The notebook uses the supplied embeddings and local descriptors, with no image-network training.

In [1]:
import os
import re
from multiprocessing import Pool

import cv2
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


from pathlib import Path
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits
from hashlib import sha256
np.random.seed(2026)
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Processed_Data/master_dataset.csv').exists())
processed = root/'Processed_Data'
meta = pd.read_csv(processed/'master_dataset.csv')
assert meta.id.is_unique and set(meta.split) == {'train','test'}
train = meta[meta.split=='train'].reset_index(drop=True)
test = meta[meta.split=='test'].reset_index(drop=True)
rows = pd.concat([train,test], ignore_index=True)
features = np.load(processed/'miewid_features.npy', mmap_mode='r')
idx = rows.miewid_index.to_numpy(int)
assert np.array_equal(np.sort(idx), np.arange(len(features)))
X = features[idx].copy()
X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
IDS, SP, Y = rows.id.to_numpy(), rows.species.to_numpy(), rows.identity.to_numpy(object)
NTR = len(train)
DATA = str(root/'DATA/reid_student_data_main')
OUT = str(root/'Models/Emad'/('cross_flank_' + sha256((' '.join(IDS)).encode()).hexdigest()[:12]))
N_SIMS, FOLDS, ROUNDS = 32, 8, 3        # simulated splits, stacking folds, stacking rounds
K, NEIGHBOURS = 15, 150                 # candidates per query, SIFT pairs verified per image
MATCH_RATIO, RANSAC_PX = 0.85, 15.0         # SIFT matching: ratio test, RANSAC pixel tolerance
SIFT_PULL = 0.05                        # weight of SIFT evidence when selecting candidates
FLANK = np.array([0, 1, 2, 2, 0, 1, 0, 1, 2, 2])    # left-ish, right-ish, other (per VIEWS order)
VIEWS = ["left", "right", "front", "back", "frontleft", "frontright", "backleft", "backright", "up", "down"]


VIEW = rows.viewpoint.map(dict(zip(VIEWS, range(len(VIEWS))))).fillna(-1).to_numpy(int)
side = np.where(VIEW >= 0, FLANK[VIEW.clip(0)], 2)
print(len(train), 'gallery images;', len(test), 'competition queries')

7163 gallery images; 2791 competition queries


In [2]:
def opposite_scores(gi, qi):
    scores = np.full((len(qi),len(gi)), np.nan, dtype=np.float32)
    for src, dst in [(0,1),(1,0)]:
        left = gi[side[gi]==src]
        right = gi[side[gi]==dst]
        paired = np.intersect1d(Y[left], Y[right])
        qr = np.flatnonzero(side[qi]==src)
        gr = np.flatnonzero(side[gi]==dst)
        if len(paired)<2 or not len(qr) or not len(gr):
            continue
        a,b = [np.stack([X[ix[Y[ix]==name]].mean(0) for name in paired]) for ix in (left,right)]
        model = Ridge(alpha=1.).fit(a,b-a)
        mapped = X[qi[qr]] + model.predict(X[qi[qr]])
        mapped /= np.maximum(np.linalg.norm(mapped,axis=1,keepdims=True),1e-12)
        scores[np.ix_(qr,gr)] = mapped @ X[gi[gr]].T
    return scores

In [3]:
def verify(pair):
    """(mutual ratio-test matches, RANSAC inliers) between two images."""
    a, b = pair
    if len(DESC[a]) < 4 or len(DESC[b]) < 4:
        return 0, 0
    bf = cv2.BFMatcher()
    back = np.array([m.trainIdx for m in bf.match(DESC[b], DESC[a])])           # nearest descriptor of b in a
    good = [m[0] for m in bf.knnMatch(DESC[a], DESC[b], k=2)
            if len(m) == 2 and m[0].distance < MATCH_RATIO * m[1].distance and back[m[0].trainIdx] == m[0].queryIdx]
    if len(good) < 4:
        return len(good), 0
    src = np.float32([XY[a][m.queryIdx] for m in good])
    dst = np.float32([XY[b][m.trainIdx] for m in good])
    _, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_PX)
    return len(good), 0 if inliers is None else int(inliers.sum())




In [4]:
def unit(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)


def embed(gallery, labels, queries):
    pca = PCA(min(384, len(gallery) + len(queries) - 1)).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too
    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)
    project = lambda a: unit(lda.transform(pca.transform(a)))
    return project(gallery), project(queries)


def per_identity(sim, inv, n):
    """Per query: max and mean-of-top-2 similarity to each identity's gallery images."""
    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))
    for j in range(n):
        s = sim[:, inv == j]
        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)
    return best, top2


def species_features(gi, qi, sp):
    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""
    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]
    Lg, Lq = embed(Rg, yg, Rq)
    ids, inv = np.unique(yg, return_inverse=True)
    n, nq = len(ids), len(qi)
    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T
    lda1, lda2 = per_identity(sim_l, inv, n)
    raw1, raw2 = per_identity(sim_r, inv, n)
    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))
    cen = Lq @ centroid.T
    verify_neighbors(gi, qi, sim_l, sim_r)
    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)
    sift1, sift2 = per_identity(inl_g, inv, n)
    good1, _ = per_identity(good_g, inv, n)
    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity
    same_view = np.full((nq, n), np.nan)
    for j in range(n):
        for v in np.unique(vq):
            cols = np.where((inv == j) & (vg == v))[0]
            if len(cols):
                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)
    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched
    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)
    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)
    seen = vq >= 0
    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)
    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)
    # transductive: neighbours among the queries themselves
    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)
    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)
    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]
    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)
    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)
    t_r = -np.sort(-raw1, 1)[:, :3]
    t_s = -np.sort(-sift1, 1)[:, :3]
    g_top = -np.sort(-sim_l, 1)
    q = pd.DataFrame({
        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),
        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],
        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],
        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),
        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),
        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),
        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],
        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),
        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),
        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],
    })
    for k in range(5):
        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]
    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence
    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]
    opposite = opposite_scores(gi,qi)
    best_opp, _ = per_identity(np.nan_to_num(opposite,nan=-2.),inv,n)
    available = best_opp.copy()
    np.put_along_axis(available,cand[:,:-1],-3.,axis=1)
    extra = available.argmax(1)
    cand[:,-1] = np.where(available.max(1)>-1,extra,cand[:,-1])
    r, c = np.repeat(np.arange(nq), K), cand.ravel()
    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)
    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)
    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]
    cd = pd.DataFrame({
        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],
        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],
        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],
        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],
        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),
        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],
    })
    cd["mapped"] = np.where(best_opp[r,c]>-2,best_opp[r,c],np.nan)
    return q, cd


def build_frames(gi, qi):
    qs, cs, order, off = [], [], [], 0
    for sp in ("hyena", "leopard"):
        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]
        q, c = species_features(g, qq, sp)
        c["q"] += off
        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off
        qs.append(q), cs.append(c), order.append(qq)
        off += len(qq)
    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))


# ----------------------------------------------------------------------------- open-set simulation
# Near-duplicate linking (raw similarity, SIFT inliers) per regime; None = random image holdout.
REGIMES = [None, (0.95, 111), (0.99, 89), (0.9, 67)]


def duplicate_groups(ix, link):
    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""
    if link is None:
        return np.arange(len(ix))
    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])
    return connected_components(sparse.csr_matrix(edges))[1]


def simulate(seed):
    """Identity-disjoint open-set split mimicking the test set: unknown identities, capped queries per
    identity, and (in most regimes) whole near-duplicate groups held out together."""
    rng, link = np.random.default_rng(seed), REGIMES[seed % len(REGIMES)]
    gallery, known, unknown = [], [], []
    for sp in ("hyena", "leopard"):
        p_unknown, frac, cap = rng.uniform(.2, .35), rng.uniform(.3, .5), int(rng.integers(12, 33))
        d = train[train.species == sp]
        names = np.array(sorted(d.identity.unique()))
        rng.shuffle(names)
        new = set(names[:int(round(p_unknown * len(names)))])
        for name, g in d.groupby("identity"):
            ix = g.index.to_numpy().copy()
            rng.shuffle(ix)
            if name in new:
                unknown += list(ix[:cap])
            elif len(ix) < 2:
                gallery += list(ix)
            else:
                n_q, held = min(max(int(round(frac * len(ix))), 1), cap, len(ix) - 1), []
                groups = duplicate_groups(ix, link)
                for c in rng.permutation(np.unique(groups)):
                    members = list(ix[groups == c])
                    if len(held) < n_q and len(held) + len(members) < len(ix):
                        held += members
                known += held[:cap]
                gallery += [i for i in ix if i not in set(held)]
    frames = build_frames(np.array(gallery), np.array(known + unknown))
    truth = np.where(np.isin(frames["order"], unknown), "new_individual", Y[frames["order"]])
    frames["truth"] = truth
    frames["q"]["unk"] = (truth == "new_individual").astype(int)
    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) == truth[frames["c"].q.to_numpy()]).astype(int)
    return frames


# ----------------------------------------------------------------------------- stacked LightGBM
PARAMS = dict(objective="binary", learning_rate=.05, num_leaves=31, min_child_samples=30, subsample=.8,
              subsample_freq=1, colsample_bytree=.8, n_estimators=400, verbose=-1)
INDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\d$")


def columns(s):
    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]
    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]
    return qf, cf


def candidate_matrix(s, qf, cf):
    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)


def fit(sims):
    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""
    qf, cf = columns(sims[0])
    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0
    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])
    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])
    ck = np.concatenate([known(s) for s in sims])
    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(
        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))
    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)


def predict(models, s):
    detector, closed, anyq, (qf, cf) = models
    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)
    p_new = detector.predict_proba(s["q"][qf])[:, 1]
    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)
    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)
    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)
    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)


def rank(s, r):
    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""
    nq = len(s["q"])
    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)
    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)
    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)


def map5(preds, truth):
    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])


def augment(s, r):
    """Add the previous round's predictions for each query and for its nearest query neighbours."""
    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])
    codes = pd.factorize(c.ident)[0].reshape(nq, K)
    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)
    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()
    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours
        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []
        for k in range(5):
            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale
            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)
            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)
            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))
            support += wk[:, None] * agree
            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))
        pb, w = np.stack(pb, 1), np.stack(w, 1)
        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)
        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]
        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()
    return dict(s, q=q, c=c)

In [5]:

from functools import lru_cache
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
cv2.setNumThreads(1)
@lru_cache(maxsize=3000)
def descriptor(i):
    path = processed/'sift_features'/IDS[i]
    d = np.load(str(path)+'_desc.npy').astype(np.float32)
    return np.load(str(path)+'_xy.npy'),np.sqrt(d/np.maximum(d.sum(1,keepdims=True),1))
class Component:
    def __init__(self,index): self.index=index
    def __getitem__(self,i): return descriptor(i)[self.index]
XY,DESC = Component(0),Component(1)
pair_cache = {}
GOOD = sparse.csr_matrix((len(X),len(X)),dtype=np.int32)
INL = GOOD.copy()
def verify_neighbors(gi,qi,lda,raw):
    global GOOD,INL
    for sims in (lda,raw):
        k=min(3,len(gi))
        for q,ns in zip(qi,np.argpartition(-sims,k-1,axis=1)[:,:k]):
            for g in gi[ns]:
                pair=tuple(sorted((q,g)))
                if pair not in pair_cache: pair_cache[pair]=verify(pair)
    a,b=np.asarray(list(pair_cache)).T
    good,inl=np.asarray(list(pair_cache.values())).T
    def matrix(v):
        return sparse.csr_matrix((np.r_[v,v],(np.r_[a,b],np.r_[b,a])),shape=(len(X),len(X)))
    GOOD,INL=matrix(good),matrix(inl)

def rank_data(frame):
    q=frame['q']
    cols,cf=columns(frame)
    cand=frame['c'][cf].copy()
    cand['unknown_candidate']=0
    blanks=pd.DataFrame(np.nan,index=q.index,columns=cf)
    blanks['unknown_candidate']=1
    values=pd.concat([cand,blanks],ignore_index=True)
    indexes=np.r_[frame['c'].q,np.arange(len(q))]
    order=np.argsort(indexes,kind='stable')
    values=values.iloc[order].reset_index(drop=True)
    return pd.concat([values,q[cols].iloc[indexes[order]].reset_index(drop=True)],axis=1)

def labels(frame):
    names=frame['c'].ident.to_numpy().reshape(-1,K)
    return np.column_stack((names,np.full(len(names),'new_individual',object)))

def domain(q):
    v=pd.get_dummies(pd.Categorical(q.view,categories=np.arange(-1,10)),dtype=float)
    v.columns=['view_'+str(i) for i in v.columns]
    return pd.concat([q[['sp','l1','lm12','r1','rm12','s1','s2']].reset_index(drop=True),v],axis=1)

def run(episodes,target):
    frames=[rank_data(e) for e in episodes]
    truth=np.concatenate([(labels(e)==e['truth'][:,None]).ravel() for e in episodes])
    ds=pd.concat([domain(e['q']) for e in episodes],ignore_index=True)
    dt=domain(target['q'])
    design=pd.concat([ds,dt],ignore_index=True)
    y=np.r_[np.zeros(len(ds)),np.ones(len(dt))]
    groups=np.concatenate([e['order'] for e in episodes]+[target['order']])
    prob=np.empty(len(y))
    for tr,va in StratifiedGroupKFold(4,shuffle=True,random_state=2026).split(design,y,groups):
        clf=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=500))
        clf.fit(design.iloc[tr],y[tr])
        prob[va]=clf.predict_proba(design.iloc[va])[:,1]
    auc=roc_auc_score(y,prob)
    odds=prob[:len(ds)]/np.maximum(1e-6,1-prob[:len(ds)])
    w=np.clip(odds,0.5,2) if auc>0.55 else np.ones_like(odds)
    model=lgb.LGBMRanker(objective='lambdarank',n_estimators=180,num_leaves=15,
                         min_child_samples=60,reg_lambda=3,n_jobs=4,verbosity=-1,random_state=2026)
    model.fit(pd.concat(frames),truth.astype(int),group=np.full(len(ds),K+1),
              sample_weight=np.repeat(w,K+1))
    score=model.predict(rank_data(target)).reshape(-1,K+1)
    print('domain AUC',round(auc,4),'mean weight',round(w.mean(),3))
    return np.take_along_axis(labels(target),np.argsort(-score,axis=1)[:,:5],axis=1)


## Local evaluation and final inference
Three quarters of the identities per species train the pilot. The remaining quarter supplies one evaluation episode. Both scorers use the same retrieved candidates, cache and gallery. The control uses 3d’s first-round probability fusion. Local MAP@5 compares the scorers under this reduced verification budget. The production fit uses all training identities in six episodes. Final output follows the supplied sample’s image order.

In [6]:

original=train.copy()
rng=np.random.default_rng(777)
dev=[]
for sp,group in original.groupby('species'):
    names=rng.permutation(group.identity.unique())
    dev.extend(group.index[group.identity.isin(names[:3*len(names)//4])])
PARAMS['n_jobs']=4
def episodes_for(subset,seeds):
    global train, GOOD, INL
    train=subset
    episodes=[]
    for seed in seeds:
        # Reset exposed pairs across independent episodes; keep verification results cached.
        pair_cache.clear()
        GOOD=sparse.csr_matrix((len(X),len(X)),dtype=np.int32)
        INL=GOOD.copy()
        episodes.append(simulate(seed))
    return episodes
with threadpool_limits(limits=4):
    dev_eps=episodes_for(original.loc[dev],range(4))
    held=episodes_for(original.drop(dev),[2026])[0]
    control=rank(held,predict(fit(dev_eps),held))
    improved=run(dev_eps,held)
    for name,pred in [('first-round control',control),('joint ranker',improved)]:
        print(name,'local MAP@5',map5(pred,held['truth']), 'queries',len(pred))
    all_eps=episodes_for(original,range(6))
    pair_cache.clear()
    target=build_frames(np.arange(NTR),np.arange(NTR,len(X)))
    predictions=run(all_eps,target)
sample=pd.read_csv(Path(DATA)/'sample_submission.csv')
submission=pd.DataFrame({'id':IDS[target['order']], 'predictions':[' '.join(r) for r in predictions]})
assert set(sample.id)==set(submission.id) and sample.id.is_unique
submission=sample[['id']].merge(submission,validate='one_to_one')
allowed=original.groupby('species').identity.apply(set)
for row in submission.merge(test[['id','species']],validate='one_to_one').itertuples():
    words=row.predictions.split()
    assert len(words)==len(set(words))==5 and set(words)<=allowed[row.species]|{'new_individual'}
output=root/'Models/Emad/submission_model6_joint.csv'
submission.to_csv(output,index=False)
print(output,len(submission),'rows')


domain AUC 0.8802 mean weight 0.764
first-round control local MAP@5 0.9174512601046126 queries 701
joint ranker local MAP@5 0.915311459819306 queries 701


domain AUC 0.6213 mean weight 0.99
/media/emadtariqu/Storage/Linux/ML_PROJECT/Models/Emad/submission_model6_joint.csv 2791 rows


## Interpretation
The control and challenger share the same six-neighbour verification budget and modified shortlist. This paired local experiment cannot establish a gain over the full three-round 3d submission. Domain AUC measures whether the logistic classifier can distinguish simulated from target queries. The competition CSV has 2,791 validated rows. Kaggle scoring is pending submission by the team.